# Challenge 1 · Superposition Under the Microscope
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hastikacheddy/qiskit-fall-fest-participant/blob/main/challenges/challenge_01_superposition/challenge_01_superposition.ipynb)

**Tier 1 - Foundations** · **Estimated time: 15-25 minutes** (an estimate, not a guarantee) · *Participant notebook*

> A lab technician needs a one-qubit source that, when measured, gives 0 or 1 with equal probability - and wants to understand exactly what the *amplitudes* behind those probabilities are.

### Learning objectives
- Build a one-qubit circuit with Qiskit and simulate it exactly.
- Explain the difference between amplitudes (complex) and measurement probabilities (real, Born rule).
- Distinguish exact probabilities from finite-shot frequencies.
- Know that different phases can hide behind identical probabilities.

### Prerequisites
- Basic Python (functions, lists, NumPy arrays)
- What a qubit is: the states |0> and |1>
- No prior Qiskit experience required

### What you will write
| Function | Status |
|---|---|
| `superposition_circuit` | required |
| `born_probabilities` | required |
| `exact_probabilities` | required |
| `circuit_for_probability` | bonus (optional) |
| `circuit_for_phase_state` | bonus (optional) |

### Acceptance criteria
Your solution is accepted when all **core** and **edge-case** requirements hold:
- `superposition_circuit()` returns a valid one-qubit circuit (no measurement or classical control).
- The circuit's exact outcome probabilities are (1/2, 1/2); every equivalent construction is accepted.
- `born_probabilities` applies the Born rule |amplitude|^2 to valid amplitude vectors of any length 2^n.
- `born_probabilities` rejects invalid input (not normalised, bad length or shape, non-finite) and never modifies its argument.
- `exact_probabilities` returns the exact outcome distribution of an n-qubit circuit in Qiskit bit order.
- `exact_probabilities` rejects circuits that contain measurements.

**Bonus (optional, never required to pass):**
- `circuit_for_probability(p0)` prepares any distribution with P(0) = p0 (non-uniform targets).
- `circuit_for_phase_state(p0, phi)` prepares the requested relative phase (same Z probabilities, different X/Y expectations).

**How this notebook works.** Run the cells from top to bottom. Cells marked **TODO** are yours. After each part run the **public checks**: they tell you what works and what does not. The instructors run additional hidden tests, so also try your own edge cases. Hints go from subtle to explicit - open them one at a time.

In [ ]:
# Setup - safe to re-run (works in Google Colab and in a local Jupyter session)
import importlib, importlib.util, pathlib, subprocess, sys

# Colab opens a notebook straight from GitHub without the project next to it: the project is fetched from this repository.
PROJECT_REPOSITORY = 'https://github.com/hastikacheddy/qiskit-fall-fest-participant.git'

if importlib.util.find_spec('qiskit_fest') is None:
    here = pathlib.Path.cwd().resolve()
    # the project is the working folder, one of its parents, or (Colab: working folder /content) a folder right below it
    candidates = [here, *here.parents, *sorted(p for p in here.iterdir() if p.is_dir())]
    root = next((p for p in candidates
                 if (p / 'pyproject.toml').is_file() and (p / 'src' / 'qiskit_fest').is_dir()), None)
    try:
        on_colab = importlib.util.find_spec('google.colab') is not None
    except ModuleNotFoundError:  # there is no `google` package at all
        on_colab = False
    if root is None and on_colab:
        # the notebook is here, the project is not: fetch it (a few seconds)
        root = here / PROJECT_REPOSITORY.rsplit('/', 1)[-1].removesuffix('.git')
        if not root.exists():
            print('Fetching the project from', PROJECT_REPOSITORY, '...')
            try:
                subprocess.check_call(['git', 'clone', '--quiet', '--depth', '1', PROJECT_REPOSITORY, str(root)])
            except (subprocess.CalledProcessError, OSError) as error:
                raise RuntimeError(f'Could not fetch the project from {PROJECT_REPOSITORY} ({error}). '
                                   "Upload the project yourself instead (see README.md, section 'Colab').") from None
    if root is None or not (root / 'pyproject.toml').is_file():
        raise RuntimeError('Open this notebook from inside the project folder (the one that contains pyproject.toml). '
                           'If you only have the notebook, download the whole project from ' + PROJECT_REPOSITORY.removesuffix('.git') +
                           " (Code > Download ZIP) or open the notebook in Google Colab - see README.md, section 'Colab'.")
    print('Installing the project and its libraries - about a minute the first time ...')
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(root)])
    except subprocess.CalledProcessError:
        print('pip could not install the project (no internet?) - continuing with the libraries that are already installed.')
    # a running kernel does not re-read the .pth file that pip has just written, so make the package importable here and now
    sys.path.insert(0, str(root / 'src'))
    importlib.invalidate_caches()

from qiskit_fest.checks import run_public_checks, if_ready
from qiskit_fest.env import print_environment
print_environment()

## The problem
You will build and analyse the simplest interesting quantum object: **one qubit in an equal superposition**.

1. **Build** a one-qubit circuit that gives `0` and `1` with probability ½ each when measured. Every correct construction is accepted.
2. **Convert amplitudes to probabilities** with the Born rule.
3. **Compute exact probabilities** of any circuit by simulating its state vector - *not* by sampling.
4. *(Bonus)* Hit a **non-uniform** target such as P(0) = 0.2, and show that identical probabilities can hide different states.

### Function contracts
| Function | Input | Output | Must raise `ValueError` when |
|---|---|---|---|
| `superposition_circuit()` | none | `QuantumCircuit` on 1 qubit, no measurements, no classical bits | - |
| `born_probabilities(amplitudes)` | 1-D sequence or array of complex amplitudes, length 2^n (n ≥ 1) | `numpy.ndarray` of floats, same length, entries ≥ 0, sum 1 | the vector is not normalised (‖a‖ differs from 1 by more than 1e-9), the length is not a power of two ≥ 2, the input is not 1-D, or contains NaN/inf. **Must not modify its input.** |
| `exact_probabilities(circuit)` | measurement-free `QuantumCircuit` on n qubits | `numpy.ndarray`, length 2^n; entry *i* is the probability of the bit-string whose integer value is *i* (qubit 0 is the **least** significant bit) | the circuit contains measurements |
| *(bonus)* `circuit_for_probability(p0)` | `p0` in [0, 1] | 1-qubit circuit with P(0) = p0 | `p0` is outside [0, 1] or not finite |
| *(bonus)* `circuit_for_phase_state(p0, phi)` | `p0` in [0, 1], real `phi` | 1-qubit circuit preparing √p0 &#124;0⟩ + e^{iφ} √(1-p0) &#124;1⟩ (up to global phase) | `p0` outside [0, 1] or an argument is not finite |

**Tolerances.** Exact quantities are compared with absolute tolerance **1e-9**. Finite-shot frequencies are never used as the only evidence of correctness.

## Concept: amplitudes versus probabilities
A single-qubit state is $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ with **complex amplitudes** $\alpha,\beta$ and $|\alpha|^2+|\beta|^2 = 1$.
Measuring in the computational basis returns `0` with probability $|\alpha|^2$ and `1` with probability $|\beta|^2$ (the **Born rule**).

* Amplitudes can be negative or complex; **probabilities are real and non-negative**. Squaring the magnitude throws the phase away.
* A *global* phase $e^{i\gamma}$ multiplies both amplitudes and is physically unobservable. A *relative* phase ($\alpha|0\rangle+e^{i\varphi}\beta|1\rangle$) changes the state - but not the Z-measurement statistics.
* **Exact** probabilities come from the state vector. **Finite-shot** frequencies come from repeating the experiment and fluctuate by about $\sqrt{p(1-p)/N}$.

The picture below is already complete (no TODO): three *different* states with identical Z-statistics.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

In [ ]:
from qiskit_fest.challenges.c01_superposition import (
    EXAMPLE_AMPLITUDES, bloch_vector, plot_exact_vs_sampled, plot_same_probabilities_different_states, shot_noise)
from qiskit_fest.viz import draw_circuit, plot_bloch

plus = QuantumCircuit(1); plus.h(0)                       # (|0> + |1>)/sqrt(2)
minus = QuantumCircuit(1); minus.x(0); minus.h(0)         # (|0> - |1>)/sqrt(2)
plus_i = QuantumCircuit(1); plus_i.h(0); plus_i.s(0)      # (|0> + i|1>)/sqrt(2)
plot_same_probabilities_different_states({"|+>": plus, "|->": minus, "|+i>": plus_i})

## Part 1 · The equal superposition  *(TODO)*
Complete `superposition_circuit()`: return a **one-qubit** `QuantumCircuit` whose measurement outcomes are `0` and `1` with probability ½ each.
Do **not** add measurements - we will look at the state vector.

<details><summary>💡 Hint 1 (subtle)</summary>

You need a gate that turns the definite state |0⟩ into a state that is "half 0, half 1". Look at the single-qubit gates `h`, `rx`, `ry`.
</details>

<details><summary>💡 Hint 2</summary>

The Hadamard gate sends |0⟩ to (|0⟩ + |1⟩)/√2. Other gates (a quarter-turn rotation about X or Y) also work - the checks accept any correct construction.
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

Create `QuantumCircuit(1)`, call one gate method on qubit `0`, and return the circuit.
</details>

In [ ]:
def superposition_circuit() -> QuantumCircuit:
    """Return a 1-qubit circuit (no measurements) whose outcomes 0 and 1 each have probability 1/2."""
    qc = QuantumCircuit(1)
    # TODO: add the gate that turns |0> into an equal superposition.
    raise NotImplementedError("TODO: implement superposition_circuit()")

In [ ]:
with if_ready():
    qc = superposition_circuit()
    draw_circuit(qc, "Your circuit")
    plot_bloch(bloch_vector(qc), "Bloch vector of your state")

In [ ]:
report = run_public_checks(1, globals(), keyword="superposition_circuit")

## Part 2 · From amplitudes to probabilities  *(TODO)*
Complete `born_probabilities(amplitudes)`.

* Accept any 1-D sequence of complex numbers of length 2, 4, 8, ... and return a float array of probabilities.
* Raise `ValueError` for invalid input (see the contract table above).
* **Never modify the caller's array.**

Worked example: the amplitudes (0.6, 0.8i) must give probabilities (0.36, 0.64).

<details><summary>💡 Hint 1 (subtle)</summary>

What is the probability that corresponds to a *complex* amplitude? Think about the length of the complex number, not its real part.
</details>

<details><summary>💡 Hint 2</summary>

Probability = (absolute value of the amplitude) squared. `np.abs` works on complex numbers. A valid state also has total probability 1 - decide how to test that with a tolerance, not with `==`.
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

Skeleton: convert with `np.asarray(..., dtype=complex)` (this makes a *copy* when the input is a list, but not for a complex array - do not assign into it!); check `ndim`, finiteness (`np.isfinite`), length (a power of two ≥ 2: compare against `2 ** round(np.log2(len))`), and `abs(np.linalg.norm(a) - 1) <= 1e-9`; then return the squared magnitudes.
</details>

In [ ]:
def born_probabilities(amplitudes) -> np.ndarray:
    """Born rule: probabilities |a_i|^2 of a normalised amplitude vector (length 2^n, n >= 1). Raise ValueError if invalid."""
    # TODO: validate the input (ValueError), then return the squared magnitudes as a float array.
    raise NotImplementedError("TODO: implement born_probabilities()")

In [ ]:
with if_ready():
    print("amplitudes   :", EXAMPLE_AMPLITUDES)
    print("probabilities:", born_probabilities(EXAMPLE_AMPLITUDES), "  (expected [0.36 0.64])")

In [ ]:
report = run_public_checks(1, globals(), keyword="born")

## Part 3 · Exact probabilities of a circuit  *(TODO)*
Complete `exact_probabilities(circuit)`: simulate the **state vector** of a measurement-free circuit and return the probability of every outcome.

* Entry *i* belongs to the bit-string with integer value *i*. **Qiskit orders bits little-endian**: qubit 0 is the *rightmost* character. For a 2-qubit circuit that only applies `x` to qubit 0, the answer is `[0, 1, 0, 0]` (outcome `"01"`).
* A circuit that contains a measurement has no single state vector - raise `ValueError`.

<details><summary>💡 Hint 1 (subtle)</summary>

Qiskit has a class that stores the full quantum state of a circuit. It lives in `qiskit.quantum_info`.
</details>

<details><summary>💡 Hint 2</summary>

`Statevector(circuit)` simulates a circuit exactly, and the resulting object can report outcome probabilities. First check `circuit.count_ops()` for a `"measure"` entry.
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

Skeleton: `if "measure" in circuit.count_ops(): raise ValueError(...)`; then build the `Statevector` and return its probability vector as a NumPy array.
</details>

In [ ]:
def exact_probabilities(circuit: QuantumCircuit) -> np.ndarray:
    """Exact outcome probabilities of a measurement-free circuit (index = bit-string value, qubit 0 = LSB). ValueError if it measures."""
    # TODO: reject circuits with measurements, then simulate the state vector.
    raise NotImplementedError("TODO: implement exact_probabilities()")

In [ ]:
with if_ready():
    qc = superposition_circuit()
    exact = exact_probabilities(qc)
    print("exact probabilities:", exact)
    plot_exact_vs_sampled(qc, exact)
    p = exact[0]
    print(f"With 1000 shots the observed frequency of '0' typically deviates by about {shot_noise(p, 1000):.3f} - "
          "which is why we test exact quantities, not one finite-shot sample.")

In [ ]:
report = run_public_checks(1, globals(), keyword="exact")

## Bonus · Non-uniform targets and hidden phases  *(optional)*
**(B1)** `circuit_for_probability(p0)`: a one-qubit circuit with P(0) = `p0` for *any* p0 in [0, 1] - for example P(0) = 0.2. A rotation about the Y axis, $R_Y(\theta)|0\rangle = \cos\frac\theta2|0\rangle+\sin\frac\theta2|1\rangle$, does it: which θ gives $\cos^2(\theta/2)=p_0$?

**(B2)** `circuit_for_phase_state(p0, phi)`: same Z-probabilities, but the state √p0 |0⟩ + e^{iφ} √(1-p0) |1⟩. You can verify φ with `bloch_vector`: the X and Y components change with φ while the Z component (and therefore every Z-measurement result) does not.

<details><summary>💡 Hint (B1)</summary>

Solve $\cos^2(\theta/2) = p_0$ for θ with `np.arccos`. Validate `p0` first (finite, between 0 and 1).
</details>

<details><summary>💡 Hint (B2)</summary>

Start from your B1 circuit and add a gate that multiplies only the |1⟩ amplitude by a phase (the phase gate `p`).
</details>

In [ ]:
def circuit_for_probability(p0: float) -> QuantumCircuit:
    """One-qubit circuit with P(measure 0) = p0 (ValueError unless 0 <= p0 <= 1 and finite)."""
    # TODO (bonus): rotate |0> so that cos^2(theta/2) = p0.
    raise NotImplementedError("TODO (bonus): implement circuit_for_probability()")

In [ ]:
def circuit_for_phase_state(p0: float, phi: float) -> QuantumCircuit:
    """One-qubit circuit preparing sqrt(p0)|0> + e^{i phi} sqrt(1-p0)|1> (up to global phase)."""
    # TODO (bonus): reuse circuit_for_probability and add the relative phase.
    raise NotImplementedError("TODO (bonus): implement circuit_for_phase_state()")

In [ ]:
with if_ready():
    for phi in (0.0, np.pi / 2, np.pi):
        c = circuit_for_phase_state(0.5, phi)
        print(f"phi={phi:4.2f}  P(0)={exact_probabilities(c)[0]:.3f}  Bloch vector (X, Y, Z) = "
              f"{tuple(round(v, 3) for v in bloch_vector(c))}")

In [ ]:
report = run_public_checks(1, globals(), keyword="bonus")

## Reflection
1. Two circuits give exactly the same Z-measurement statistics. Does that mean they prepare the same state? Give an example from this notebook and say which measurement would tell them apart.
2. Why is "run the circuit 1000 times and compare the counts with 0.5" a poor *correctness test*? How would you choose the number of shots and a tolerance if you had no other option?
3. A friend says the amplitudes of a qubit are "the probabilities with signs". What is wrong with that statement?

### Hand-in
When you are done, save your functions as a file (or simply hand in this notebook).

In [ ]:
# Optional: save your functions as a stand-alone file you can hand in
from qiskit_fest.export import export_submission
export_submission(globals(), 1)